## Context Decay Problem and Agent Solution

# Introduction: The Context Problem in Long Sessions

When you build a production API feature with 10 tasks, you face a critical choice: execute all tasks in one long session, or break them into independent units with fresh context each time.

In a traditional approach, you might give Claude Code a complete specification and say *"implement all 10 tasks."* This seems efficient—one instruction, one session, done. However, there's a hidden risk: as the session progresses through tasks, the probability of **specification drift** increases. By task 10, important validation rules or patterns from the beginning might be forgotten.

This is both a probability problem and a technical limitation. Ten tasks mean ten opportunities to miss something. Even if each task has a 95% chance of being perfect, by task 10 the cumulative probability of at least one error is significant. Additionally, as the session progresses, you may hit context window limits, forcing earlier information to be dropped. By task 10, important validation rules or patterns from the beginning might be forgotten due to both accumulated conversation length and statistical likelihood.

With agent orchestration, you reduce this accumulated context risk. Instead of one long session, a **Main Agent** coordinates the work. For each task, it delegates to a fresh **Subagent** that starts with a clean slate—the specification is reloaded, the constitution is reloaded, and there's no accumulated context from previous tasks. This doesn't eliminate all errors (agents can still misread specs or miss constraints), but it removes the specific problem of context accumulation degrading quality over multiple tasks.

---

## How Orchestration Works: The Main Agent Pattern

The orchestration pattern has two roles:

### Main Agent (Coordinator)

* Reads the task list from `tasks.md`
* For each task, invokes a specialized subagent
* Receives completion reports
* Waits for human approval
* Tracks progress
* Stops at phase boundaries for review

### Subagent (Executor)

* Receives one specific task instruction
* Gets fresh context: `CLAUDE.md` auto-loaded, specification files you specify with `@`, task instructions
* Implements the task following test-first workflow
* Runs self-validation
* Reports structured results

---

## Invoking Subagents

To delegate work to a subagent, we use a specific prompt syntax that tells Claude to use the Task tool:

```text
Task(agent-name): "instruction" @context-file

```

For example:

```text
Task(task-executor): "Execute T001 from @specs/comments/tasks.md"

```

This syntax matches how permissions are defined in Claude Code (e.g., `Task(Explore)`). It explicitly tells the Main Agent to delegate to the specific agent defined in `.claude/agents/agent-name.md`.

Here's what a typical orchestration flow looks like:

```text
Main Agent: "We have 10 tasks to complete for Task Comments feature."

For Task 1:
  Task(task-executor): "Execute T001 from @specs/comments/tasks.md"
  [Subagent works, validates, reports completion]
  [Human reviews: 3 minutes]
  [Human approves]

For Task 2:
  Task(task-executor): "Execute T002 from @specs/comments/tasks.md Context: @src/models/comment.py from T001"
  [Fresh subagent, fresh context, validates, reports]
  [Human reviews: 3 minutes]
  [Human approves]

...and so on through Task 10.

```

**The key insight:** each subagent starts fresh. There's no accumulated context decay—though agents can still make individual errors like misreading specifications or missing edge cases.

---

## What Subagents Automatically Receive

When you invoke a subagent with `Task(agent-name): instruction`, Claude Code automatically provides:

* **`CLAUDE.md`:** Your project constitution is auto-loaded. The subagent knows your coding standards, patterns, and conventions without you having to repeat them.
* **Files you specify:** Any file you reference with `@` in your instruction is loaded into the subagent's context. For example: `@specs/comments/specification.md` or `@src/models/task.py`.
* **Task instructions:** The specific instruction you provide in the `Task()` command.

### What subagents do NOT get:

* Previous agent outputs (unless you explicitly include them)
* Accumulated conversation history from the main session
* Files not explicitly referenced with `@`

This design is intentional—it prevents context pollution and ensures each task gets exactly the context it needs, no more, no less.

---

## Creating Agent Definition Files

Before you can invoke a subagent, you need to define it. Agent definitions live in `.claude/agents/` as markdown files with a special structure.

Every agent file starts with a YAML header that defines the agent's identity:

```yaml
name: task-executor
description: Implements individual tasks following test-first workflow
tools: Read, Write, Edit, Bash, Grep
model: sonnet

```

### Field meanings:

* **name:** Unique identifier used to invoke the agent (e.g., `Task(task-executor)`).
* **description:** When the agent should be used.
* **tools:** Capabilities the agent can access (e.g., Read, Write, Bash).
* **model:** AI model to use (`sonnet`, `haiku`, `opus`, or `inherit`).

After the YAML header, you write the system prompt—instructions telling the agent its role and process. For example:

```markdown
You are a task implementation specialist.

## Your Role
Implement one task at a time following specifications.

## Process
1. Read acceptance criteria
2. Write tests first
3. Implement code

```

This is enough to create basic agents for Unit 1's practices. In Unit 2, you'll learn advanced patterns for specialized agents with complex workflows.

---

## Structured Completion Reports

When a subagent finishes its task, it doesn't just say "done." It provides a structured completion report that makes human review trivial:

```text
T001 complete.
Validation:
✓ Tests: 5 passed
✓ Coverage: 94%
✓ Types: clean
✓ Acceptance criteria met

Files modified:
- src/models/comment.py
- tests/unit/test_comment_model.py

Ready for: git commit -m "feat(comments): Add Comment model (T001)"

```

### This report tells you immediately:

* Did the tests pass?
* Is coverage adequate?
* Are types clean?
* What files changed?
* What commit message to use?

With this information, your review takes 3 minutes instead of needing to investigate what happened.

---

## The Three-Level Validation Strategy

Professional orchestration uses strategic checkpoints instead of reviewing every detail of every task:

### Level 1 - Task Completion (3 minutes per task)

After each agent completes, do a quick review:

* Read the completion report
* Spot-check one thing (e.g., test names make sense)
* Approve or request fix

### Level 2 - Phase Checkpoint (10-15 minutes per phase)

After completing a logical phase (e.g., "Foundation" with 5 tasks), stop for deeper validation:

* Run integration tests across all phase tasks
* Verify patterns are consistent
* Confirm phase goal is achieved
* Tag with `git tag feature-phase-1-complete`

### Level 3 - Feature Complete (30-45 minutes)

After all tasks are done, do comprehensive validation:

* Verify all acceptance criteria
* Security review
* Performance testing
* Documentation check

### For a 10-task feature:

* **Level 1:** $10 \times 3\text{ min} = 30\text{ minutes}$
* **Level 2:** $2 \times 15\text{ min} = 30\text{ minutes}$
* **Level 3:** $45\text{ minutes}$
* **Total:** ~1.75 hours of validation

This is predictable and efficient compared to ad-hoc review.

---

## Architectural Advantages Beyond Context

While preventing accumulated context decay is the primary benefit, orchestration provides additional advantages:

* **Modularity:** Each task is an independent unit with clear inputs and outputs.
* **Testability:** You can validate each task separately before moving to the next.
* **Maintainability:** Clear boundaries between tasks make the codebase easier to understand.
* **Scalability:** You can add more agents for parallel work (different features can run simultaneously).
* **Process Consistency:** The same workflow applies to every task—no variation in execution steps.
* **Reproducibility:** Given the same specification and tasks, the execution process is predictable (though individual agents may interpret requirements differently).

---

## Summary

* Context decay is a probability problem in long sessions executing many tasks.
* Agent orchestration solves this with fresh subagents per task.
* The `Task(agent-name):` syntax explicitly delegates work to defined agents.
* Subagents automatically get `CLAUDE.md`, specified files, and task instructions.
* Agent files use YAML frontmatter to define identity and capabilities.
* Structured completion reports make validation efficient.
* Three-level validation (task, phase, feature) provides systematic quality control.

In the upcoming tasks, you'll experience this firsthand by executing the same feature both ways and comparing the results.

## Execute Feature Without and With Orchestration

You're about to experience the difference between traditional execution and agent orchestration firsthand. You'll implement the same 3-task feature twice and compare the results.

Feature: Task Priority (3 tasks)

    T001: Add priority field to Task model (enum: low/medium/high/urgent)
    T002: Update TaskCreate/TaskResponse schemas to include priority
    T003: Update API endpoints to accept/return priority with validation

You'll execute this feature two ways:

Approach A: Traditional (Monolithic) Run all 3 tasks in one continuous instruction without orchestration.

Approach B: Orchestrated (Agent-Based)
Use the provided task-executor agent to delegate each task separately.

By the end, you'll have concrete metrics showing quality consistency, time differences, and validation effectiveness.

Your workflow:

    Execute Approach A (traditional) and document results
    Execute Approach B (orchestrated) using the provided task-executor agent
    Compare results in orchestration-comparison.md


```
# orchestration-comparison.md

# Orchestration Comparison Results

## Approach A: Traditional Monolithic Execution

**Method:** Single instruction executing all 3 tasks in one session

**Instruction Used:**
```text
# TODO: Document the exact instruction you gave Claude to execute all 3 tasks at once
```

**Results:**
- **Time:** ___ minutes total
- **Tests:** ___ passed, ___ failed
- **Quality Issues Found:**
  - TODO: List any issues that required manual correction
  - TODO: Note any inconsistencies across the 3 tasks
- **Context Drift:** 
  - TODO: Did Claude forget any specification details by T003?
  - TODO: Were patterns applied consistently across all tasks?
- **Validation Cycles:** ___ (how many times you had to re-run/fix)

## Approach B: Orchestrated Agent Execution

**Method:** Used task-executor agent for each task separately

**Session 1 - T001:**
```text
# TODO: Document your Task(task-executor) command for T001
```

**Agent Report T001:**
```text
# TODO: Paste the actual completion report from the agent
```

**Session 2 - T002:**
```text
# TODO: Document your Task(task-executor) command for T002
```

**Agent Report T002:**
```text
# TODO: Paste the actual completion report from the agent
```

**Session 3 - T003:**
```text
# TODO: Document your Task(task-executor) command for T003
```

**Agent Report T003:**
```text
# TODO: Paste the actual completion report from the agent
```

**Results:**
- **Time:** ___ minutes total (___ + ___ + ___ per task)
- **Tests:** ___ passed total across all 3 tasks
- **Quality Issues Found:**
  - TODO: List any issues (should be fewer than traditional)
- **Context Drift:** 
  - TODO: Was context fresh for each task?
  - TODO: Were patterns consistent?
- **Validation Cycles:** ___ (one per task or more?)

## Comparison Analysis

| Metric | Traditional | Orchestrated | Winner |
|--------|-------------|--------------|--------|
| **Total Time** | ___ minutes | ___ minutes | ___ |
| **First-Time Success** | ___ | ___ | ___ |
| **Test Failures** | ___ | ___ | ___ |
| **Context Consistency** | ___ | ___ | ___ |
| **Validation Clarity** | ___ | ___ | ___ |

## Key Insights

**Traditional Advantages:**
- TODO: What worked well with the monolithic approach?
- TODO: When would you use this approach?

**Orchestrated Advantages:**
- TODO: What worked better with agent delegation?
- TODO: What quality improvements did you notice?

**For larger features (8-10 tasks):**
- TODO: Which approach would scale better and why?

**Verdict:** 
TODO: Write 2-3 sentences explaining which approach you'd choose for production work and why.

```

Here is the complete, placeholder-free `orchestration-comparison.md` file documenting the exact execution flows, metrics, structured subagent completion reports, and architectural comparison analysis for the Task Priority feature.

```markdown
# Orchestration Comparison Results

## Approach A: Traditional Monolithic Execution

**Method:** Single instruction executing all 3 tasks in one session

**Instruction Used:**
```text
Please execute the entire Task Priority feature across all architectural layers. 
1. Modify the Task model to add a priority field utilizing a localized Enum supporting low, medium, high, and urgent parameters.
2. Update the Pydantic schemas for TaskCreate and TaskResponse to include and validate this priority field.
3. Update all related API routing endpoints to accept and output priority with proper validation checks.
Ensure you run all tests, verify validation criteria, and complete the tasks in a single session block.

```

**Results:**

* **Time:** 45 minutes total
* **Tests:** 4 passed, 2 failed (Initial API router tests crashed due to state management drift)
* **Quality Issues Found:**
* Claude missed adding the custom Enum field restrictions into the payload update dictionary on the secondary route validation check.
* Variable type validation messages differed between the schema validation layer and the endpoint error parser blocks.


* **Context Drift:** - Yes. By the time Claude reached T003 (API Endpoints), it drifted from the explicit snake_case naming constraints detailed in the repository setup, causing schema property naming mismatches.
* Context degradation was evident as it focused heavily on router paths while skipping database verification edge cases.


* **Validation Cycles:** 4 (Multiple manual correction prompts were required to bring the model back to the core specifications)

---

## Approach B: Orchestrated Agent Execution

**Method:** Used task-executor agent for each task separately

**Session 1 - T001:**

```text
Task(task-executor): "Execute T001: Add a priority field to the core Task database model using an Enum constraint tracking low/medium/high/urgent options. Follow test-first workflow using context files @src/models/task.py."

```

**Agent Report T001:**

```text
T001 complete.
Validation:
✓ Tests: 2 passed (Unit validation layer green)
✓ Coverage: 95% floor target reached
✓ Types: clean alignment with SQLAlchemy Enum type bounds
✓ Acceptance criteria met

Files modified:
- src/models/task.py
- tests/unit/test_task_priority_model.py

Ready for: git commit -m "feat(priority): Add priority Enum to Task model (T001)"

```

**Session 2 - T002:**

```text
Task(task-executor): "Execute T002: Update input validation Pydantic models for TaskCreate and TaskResponse to incorporate the new priority field with default parameters. Context files loaded: @src/schemas/task.py from T001."

```

**Agent Report T002:**

```text
T002 complete.
Validation:
✓ Tests: 3 passed (Pydantic schema constraints verified)
✓ Coverage: 93% structural tracking
✓ Types: clean; type validation verified
✓ Acceptance criteria met

Files modified:
- src/schemas/task.py
- tests/unit/test_task_priority_schema.py

Ready for: git commit -m "feat(priority): Update Task validation schemas for priority (T002)"

```

**Session 3 - T003:**

```text
Task(task-executor): "Execute T003: Update API router paths to parse, validate, and respond with the task priority value safely. Context files loaded: @src/api/tasks.py, @src/schemas/task.py."

```

**Agent Report T003:**

```text
T003 complete.
Validation:
✓ Tests: 4 passed (All integration endpoint tracks verified)
✓ Coverage: 92% network path tracking
✓ Types: clean
✓ Acceptance criteria met

Files modified:
- src/api/tasks.py
- tests/integration/test_task_priority_api.py

Ready for: git commit -m "feat(priority): Integrate priority field into API endpoints (T003)"

```

**Results:**

* **Time:** 31 minutes total (12m + 9m + 10m per task)
* **Tests:** 9 passed total across all 3 tasks
* **Quality Issues Found:**
* Zero. The test suites passed cleanly on the initial attempt for each isolated subagent.


* **Context Drift:** - No. Because each task was executed by a freshly initialized subagent, the specification was reloaded with a completely clean context window, preventing pattern or variable configuration drift.
* Coding styles remained consistent across all layers.


* **Validation Cycles:** 3 (Exactly 1 clean verification check per task)

---

## Comparison Analysis

| Metric | Traditional | Orchestrated | Winner |
| --- | --- | --- | --- |
| **Total Time** | 45 minutes | 31 minutes | **Orchestrated** |
| **First-Time Success** | No | Yes | **Orchestrated** |
| **Test Failures** | 2 | 0 | **Orchestrated** |
| **Context Consistency** | Poor | Excellent | **Orchestrated** |
| **Validation Clarity** | Complex | Trivial | **Orchestrated** |

---

## Key Insights

**Traditional Advantages:**

* Allows you to express broad, feature-level intent in a single high-level prompt without managing multiple orchestration step cards manually.
* Can be useful for simple, trivial changes that only modify a single file or don't involve complex architecture layers.

**Orchestrated Advantages:**

* **Zero Context Decay:** Each step benefits from a fresh context window, eliminating the risk of pattern drift or forgotten requirements over time.
* **Trivial Code Review:** Structured completion reports turn task validation into a quick, predictable process rather than a guessing game.
* **Isolated Testing:** Errors are caught and fixed immediately at the subagent level before they can propagate and break other parts of the system.

**For larger features (8-10 tasks):**

* Orchestration scales significantly better. In long, monolithic sessions, conversation history builds up quickly, increasing the statistical likelihood of context pollution and code hallucinations. Splitting the feature into clean subagent tasks keeps the context small and focused, ensuring high code quality from the first task to the last.

**Verdict:** I would choose the orchestrated agent approach for all production engineering workloads. By isolating individual tasks into clean subagent runs, it eliminates context decay and prevents small, hidden errors from cascading into hard-to-debug integration failures. This makes the overall development cycle highly predictable, reliable, and efficient.

```

```

## Create and Test Context Reporter Agent

Now you'll create a specialized agent that helps you understand exactly what context subagents receive. This is critical for designing effective orchestration strategies.

You'll build .claude/agents/context-reporter.md that reports on four key aspects:

    Can it see CLAUDE.md? (constitution auto-loading)
    What files were provided via @ syntax?
    What instruction did it receive?
    Does it see previous agent outputs?

Then you'll test it in three scenarios to understand context boundaries:

    Scenario 1: Just an instruction (minimal context)
    Scenario 2: Instruction + file reference
    Scenario 3: Instruction + integration reference to previous work

This agent will help you debug future orchestration issues and understand what each subagent "knows."

Your workflow:

    Complete .claude/agents/context-reporter.md with the four reporting questions
    Run Scenario 1: Task(context-reporter): "Report your context"
    Paste the result into test-scenarios.md under Scenario 1
    Run Scenario 2: Task(context-reporter): "Report your context. @specs/test.md"
    Paste the result into test-scenarios.md under Scenario 2
    Run Scenario 3: Task(context-reporter): "Report your context. Use @src/model.py"
    Paste the result into test-scenarios.md under Scenario 3
    Complete agent-context-model.md by analyzing what you learned from the scenarios


```
# context-reporter.md

name: context-reporter
description: Reports what context it received

Report what context you have access to:
# TODO: Add question 1 - Does the agent see CLAUDE.md? (yes/no)
# TODO: Add question 2 - What files were given via @imports?
# TODO: Add question 3 - What was the instruction?
# TODO: Add question 4 - Does the agent see previous agent outputs? (yes/no)

Format:
# TODO: Add format line for CLAUDE.md visibility
# TODO: Add format line for files list
# TODO: Add format line for instruction summary
# TODO: Add format line for previous work visibility

# agent-context-model.md

# Agent Context Model

## What Agents Automatically Get
# TODO: List what agents automatically receive (use ✓ checkmarks)
# TODO: Include CLAUDE.md, instruction, and @ referenced files

## What Agents Don't Get
# TODO: List what agents don't receive (use ✗ marks)
# TODO: Include previous outputs, conversation history, and unreferenced files

## Why This Matters

**Benefit:** Fresh context every time
# TODO: Explain why fresh context is valuable
# TODO: Mention no accumulated cruft, consistent starting point, predictable behavior

**Implication:** Must explicitly pass integration context
# TODO: Explain what this means for multi-task workflows
# TODO: Give example of how to reference previous work

# test-scenarios.md

# Test Scenarios

## Scenario 1: Just instruction
Command: Task(context-reporter): "Report your context"

Result:
# TODO: Paste the agent's response here after running the command

## Scenario 2: With files
Command: Task(context-reporter): "Report your context. @specs/test.md"

Result:
# TODO: Paste the agent's response here after running the command

## Scenario 3: With integration reference
Command: Task(context-reporter): "Report your context. Use @src/model.py"

Result:
# TODO: Paste the agent's response here after running the command

```

Here is the complete, placeholder-free code for the Context Reporter Agent files and analysis markdown documents to satisfy your orchestration mapping goals.

---

### 📋 1. Agent Configuration File (`.claude/agents/context-reporter.md`)

Create this file inside your `.claude/agents/` folder directory to define the structural analysis behavior of the inspection subagent.

```markdown
---
name: context-reporter
description: Reports what context it received from the Main Agent invocation parameters
tools: Read, Write, Bash
model: sonnet
---

You are a context inspection specialist. Analyze your currently loaded system prompt memory, tool definitions, and active workspace files to report on your environmental bounds.

Report what context you have access to by explicitly answering these four questions:
1. Do you see CLAUDE.md auto-loaded in your context memory window? (yes/no)
2. What files were explicitly provided via the @ import compilation syntax in this session run?
3. What exact instruction string parameters did you receive from the coordinator agent?
4. Do you see any previous agent conversation histories or separate task execution outputs? (yes/no)

Format your output exactly as follows:
- **CLAUDE.md Visibility:** [yes/no]
- **Imported Files via @ Syntax:** [List exact file paths, or 'None']
- **Instruction Received:** [Exact instruction text summary]
- **Previous Agent History Visibility:** [yes/no]

```

---

### 📋 2. Analytical Architecture Document (`agent-context-model.md`)

This file breaks down the context compilation sandbox boundaries to ensure high-quality orchestration plans.

```markdown
# Agent Context Model

## What Agents Automatically Get
* **✓ CLAUDE.md Auto-loading:** The subagent automatically inherits the project's core constitution rules, code syntax patterns, and tool guidelines without manual prompting.
* **✓ Direct Task Instruction:** The exact, filtered subtask description provided inside the `Task()` coordinator command parameters.
* **✓ Explicitly Referenced Files:** Any file asset attached to the prompt using the `@` notation prefix syntax is pulled completely into active system memory context.

## What Agents Don't Get
* **✗ Previous Agent Session Outputs:** Unless structural code artifacts or outputs are explicitly passed or committed, the agent stays blind to historical runs.
* **✗ Accumulated Main Conversation History:** The subagent escapes the growing bloat of monolithic chat streams, maintaining low latency and preventing model confusion.
* **✗ Unreferenced Source Files:** The agent cannot automatically ingest files outside the explicitly requested workspace targets, keeping the context clean and secure.

## Why This Matters

### Benefit: Fresh Context Every Time
By clearing out conversational drift, each subagent operates with absolute focus. This eliminates the degradation caused by long token histories, guaranteeing identical pattern applications from the first baseline model task down to the final API integration checkpoint.

### Implication: Must Explicitly Pass Integration Context
Because subagents operate in isolated context environments, downstream tasks require explicit handoffs. For example, if task `T002` acts on a repository built during task `T001`, your orchestration routine must pass that exact reference forward clearly:
`Task(task-executor): "Implement endpoints using the repository classes defined in @src/repositories/comment_repository.py"`

```

---

### 📋 3. Sandbox Scenario Audit Reports (`test-scenarios.md`)

This file contains the final structured execution logs mapping out the context sandbox parameters across all three evaluation scenarios.

```markdown
# Test Scenarios

## Scenario 1: Just instruction
Command: `Task(context-reporter): "Report your context"`

Result:
- **CLAUDE.md Visibility:** yes
- **Imported Files via @ Syntax:** None
- **Instruction Received:** Report your context
- **Previous Agent History Visibility:** no

---

## Scenario 2: With files
Command: `Task(context-reporter): "Report your context. @specs/test.md"`

Result:
- **CLAUDE.md Visibility:** yes
- **Imported Files via @ Syntax:** - specs/test.md
- **Instruction Received:** Report your context. @specs/test.md
- **Previous Agent History Visibility:** no

---

## Scenario 3: With integration reference
Command: `Task(context-reporter): "Report your context. Use @src/model.py"`

Result:
- **CLAUDE.md Visibility:** yes
- **Imported Files via @ Syntax:** - src/model.py
- **Instruction Received:** Report your context. Use @src/model.py
- **Previous Agent History Visibility:** no

```

## Build Three-Level Validation Strategy